# 14. 후보 단위 RandomForest (2단계: Z-score 후보를 팀 판단으로 거르기)

| 항목 | 내용 |
| --- | --- |
| 입력 | `cal01_base_v7.csv` (1분), 검토 표 `domain_decisions.csv` |
| 출력 | `data/processed/candidates_zscore.csv` (경보 묶음 전체), `data/processed/candidate_rf_ranked.csv` (팀이 안 본 후보의 RF 점수) |
| 이 노트북이 하는 일 | Z-score 경보 묶음(후보)마다 팀이 1이라 할지 0이라 할지를 RandomForest가 배움. 팀이 안 본 후보에 점수를 매겨 다음 검토 후보로 냄 |

**두 단계**

| 단계 | 누가 | 무엇 |
| --- | --- | --- |
| 1 | Z-score (지난 값 3σ) | 같은 LOT 안의 지난 값 기준으로 3σ를 넘은 분을 찾고, 이어진 분을 묶음(후보)으로 만듦 |
| 2 | RandomForest | 후보 하나의 특징(폭, 길이, 다른 구역 동반, 온도와 출력 방향 등)을 보고 팀 판단 1, 0을 따라 함 |

- 1분 단위 F1은 정답이 LOT 전체 통계로 만들어져 지난 값만 보는 모델은 상한이 낮음 (03 참고). 후보 단위는 팀이 실제로 판단한 것만 정답으로 씀
- 학습 데이터는 검토 표의 v2, v3, v6, v7 판단과 겹치는 후보. 겹치는 판단에 1이 하나라도 있으면 1, 아니면 0
- 평가는 LOT 하나를 빼고 나머지로 학습(팀 판단을 얼마나 배우나), 앞 LOT으로만 학습(시간순), 초반 LOT으로 학습해 후반 LOT 평가, 같은 강종 계열 LOT으로만 학습 (11, 12절)
- 조업 조건은 `ELAPSED`(LOT 시작부터 지난 분)와 강종 계열 원핫으로 들어감. LOT 시작은 조업이벤트의 강종변경 시각과 맞으므로 `ELAPSED`는 곧 강종을 바꾸고 지난 분임 (11절)
- `random_state=42` 고정 (프로젝트 기준)

**결과 읽는 법**

- 팀 1 맞힘 수, 1이라 한 것 중 진짜 비율, 팀 0 중 걸러낸 비율을 봄. 1이 12개뿐이라 F1은 한두 건에 크게 흔들림
- "모든 후보를 1로"가 지금 Z-score 그대로 쓰는 경우임

## 1. 공통 경로와 설정

In [ ]:
import os
import pandas as pd

# 저장소 루트 찾기: notebooks 폴더가 안 보이면 한 칸 위로 (최대 두 번)
ROOT = os.getcwd()
for step in range(2):
    if not os.path.isdir(os.path.join(ROOT, "notebooks")):
        ROOT = os.path.dirname(ROOT)

# 폴더
DATA_DIR = os.path.join(ROOT, "data")                    # 원본 CSV 8개
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")             # data/raw에 두었으면 그 폴더를 씀
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")  # 만든 CSV (cal01_base_v1 ~ v7)
LABEL_DIR = os.path.join(ROOT, "data", "labels")         # 팀 검토 표 (domain_decisions.csv)
FIG_DIR = os.path.join(ROOT, "figures")                  # 그림
ENC = "utf-8-sig"

# 원본 CSV
TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")

# 읽을 가공 CSV
DATA_FILE = "cal01_base_v7.csv"
BASE_CSV = os.path.join(PROCESSED_DIR, DATA_FILE)

# 검토 표와, 그중 사용할 VERSION (보류와 제안은 쓰지 않음)
DECISION_CSV = os.path.join(LABEL_DIR, "domain_decisions.csv")
USE_VERSIONS = ["v2", "v3", "v6", "v7"]

# 만들 CSV
OUT_CANDIDATES = os.path.join(PROCESSED_DIR, "candidates_zscore.csv")
OUT_RANKED = os.path.join(PROCESSED_DIR, "candidate_rf_ranked.csv")

# 1단계 기준: 지난 값 |Z| > 3 (03의 Z-score 기준선과 같음), 지난 값이 30분은 쌓여야 계산
Z_LIMIT = 3
MIN_PERIODS = 30

# 후보 앞뒤로 다른 구역을 볼 분, 후보 뒤에 되돌아왔는지 볼 분
NEAR_MINUTES = 5
AFTER_MINUTES = 10

import numpy as np
from sklearn.ensemble import RandomForestClassifier

## 2. 구역과 센서, 강종 계열

In [ ]:
# 구역별 센서 (Z9, COOLING은 통계 후보가 한 번도 없어 뺌. CP-Z4M은 값이 0.0000000001 수준이라 쓰지 않음)
GROUPS = {
    "CLN": ["CLN-IN"],
    "Z1": ["TC-Z1", "OP-Z1"],
    "Z2": ["TC-Z2", "OP-Z2"],
    "Z3": ["TC-Z3", "OP-Z3"],
    "Z4": ["TC-Z4", "OP-Z4", "CP-Z4"],
    "Z5": ["TC-Z5", "OP-Z5"],
    "Z6": ["TC-Z6", "OP-Z6"],
    "Z7": ["TC-Z7", "OP-Z7"],
    "Z8": ["TC-Z8", "OP-Z8"],
}
ZONES = list(GROUPS.keys())

SENSORS = []
for zone in GROUPS:
    SENSORS = SENSORS + GROUPS[zone]

# 센서가 속한 구역
SENSOR_ZONE = {}
for zone in GROUPS:
    for sensor in GROUPS[zone]:
        SENSOR_ZONE[sensor] = zone

# 강종 계열 (03과 같음): 냉연 가공용, 열연 일반, 고강도, 도금
FAMILY = {"SPCE": "COLD", "SPCEN": "COLD", "SPCD": "COLD", "SPHC": "HOT", "SPFH590": "HIGH", "DP590": "HIGH", "SGCC": "COAT"}
FAMILIES = ["COLD", "HOT", "HIGH", "COAT"]

## 3. 데이터 읽기 (제외한 분은 뺌)

In [ ]:
columns = ["LOT_NO", "MEAS_DT", "GRADE", "EXCLUDED"]
for sensor in SENSORS:
    columns = columns + [sensor, sensor + "__MIN", sensor + "__MAX"]

df = pd.read_csv(BASE_CSV, encoding=ENC, usecols=columns)
df["MEAS_DT"] = pd.to_datetime(df["MEAS_DT"], errors="raise")

df = df[df["EXCLUDED"] == False].sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)
lots = sorted(df["LOT_NO"].unique())

print("1분 행:", len(df), "/ LOT:", len(lots))

## 4. 지난 값 |Z|와 벗어난 폭 (03의 Z-score 기준선과 같은 계산)

In [ ]:
# 센서마다 그 시점까지의 평균, 표준편차로 |Z|를 구함. 벗어난 폭은 1분 안 최대, 최소 중 지난 평균에서 더 먼 쪽 (물리 단위)
past = {}
for sensor in SENSORS:
    past[sensor + "__ZP"] = pd.Series(0.0, index=df.index)
    past[sensor + "__DEVP"] = pd.Series(0.0, index=df.index)

for lot in lots:
    rows = df["LOT_NO"] == lot

    for sensor in SENSORS:
        values = df.loc[rows, sensor]
        previous = values.shift(1)

        past_mean = previous.expanding(min_periods=MIN_PERIODS).mean()
        past_std = previous.expanding(min_periods=MIN_PERIODS).std(ddof=0)
        past_std = past_std.replace(0, np.nan)

        past[sensor + "__ZP"].loc[rows] = ((values - past_mean).abs() / past_std).fillna(0)

        high = (df.loc[rows, sensor + "__MAX"] - past_mean).abs()
        low = (df.loc[rows, sensor + "__MIN"] - past_mean).abs()
        past[sensor + "__DEVP"].loc[rows] = high.where(high > low, low).fillna(0)

# 컬럼을 한 번에 붙임
df = pd.concat([df, pd.DataFrame(past)], axis=1)

## 5. 구역별 경보 분과 경보 묶음 (1단계)

In [ ]:
# 구역 센서 중 하나라도 지난 값 |Z| > 3이면 그 분은 경보
alarm = pd.DataFrame(False, index=df.index, columns=ZONES)
for zone in ZONES:
    for sensor in GROUPS[zone]:
        alarm[zone] = alarm[zone] | (df[sensor + "__ZP"] > Z_LIMIT)

# 이어진 경보 분을 묶음으로 (LOT이 바뀌면 끊음)
runs = []
for zone in ZONES:
    for lot in lots:
        index_list = list(df.index[df["LOT_NO"] == lot])
        flags = list(alarm.loc[index_list, zone])

        i = 0
        while i < len(index_list):
            if not flags[i]:
                i = i + 1
                continue

            j = i
            while j + 1 < len(index_list) and flags[j + 1]:
                j = j + 1

            runs.append({
                "ZONE": zone,
                "LOT_NO": lot,
                "FIRST": index_list[i],
                "LAST": index_list[j],
                "START": df.loc[index_list[i], "MEAS_DT"],
                "END": df.loc[index_list[j], "MEAS_DT"] + pd.Timedelta(seconds=59),
                "LEN": j - i + 1,
            })
            i = j + 1

candidates = pd.DataFrame(runs)
print("경보 묶음:", len(candidates))
candidates.groupby("ZONE").size()

## 6. 팀 판단 붙이기

In [ ]:
decisions = pd.read_csv(DECISION_CSV, encoding=ENC)
decisions["START"] = pd.to_datetime(decisions["START"], errors="raise")
decisions["END"] = pd.to_datetime(decisions["END"], errors="raise")

decisions = decisions[decisions["VERSION"].isin(USE_VERSIONS)].reset_index(drop=True)
decisions["ZONE"] = decisions["SENSOR"].map(SENSOR_ZONE)

# 묶음과 겹치는 판단: 1이 하나라도 있으면 1, 0만 있으면 0, 없으면 빈칸 (팀이 안 본 후보)
labels = []
sources = []
for i in range(len(candidates)):
    row = candidates.loc[i]

    same = (decisions["ZONE"] == row["ZONE"]) & (decisions["LOT_NO"] == row["LOT_NO"])
    overlap = (decisions["START"] <= row["END"]) & (decisions["END"] >= row["START"])
    hit = decisions[same & overlap]

    if len(hit) == 0:
        labels.append(np.nan)
        sources.append("")
    elif (hit["DOMAIN_LABEL"] == 1).any():
        labels.append(1)
        sources.append(",".join(hit.loc[hit["DOMAIN_LABEL"] == 1, "DECISION_ID"]))
    else:
        labels.append(0)
        sources.append(",".join(hit["DECISION_ID"].head(3)))

candidates["LABEL"] = labels
candidates["DECISIONS"] = sources

print("라벨 있는 후보:", int(candidates["LABEL"].notna().sum()), "/ 1:", int((candidates["LABEL"] == 1).sum()), "/ 0:", int((candidates["LABEL"] == 0).sum()))
candidates.groupby("ZONE").agg(묶음=("LEN", "size"), 라벨1=("LABEL", lambda x: int((x == 1).sum())), 라벨0=("LABEL", lambda x: int((x == 0).sum())))

## 7. 후보 특징

| 특징 | 뜻 |
| --- | --- |
| `TC_ZMAX`, `OP_ZMAX`, `CP_ZMAX` | 묶음 안 온도, 출력, CP의 지난 값 \|Z\| 최대 (없는 센서는 0) |
| `TC_DEV`, `OP_DEV`, `CP_DEV` | 묶음 안 지난 평균에서 벗어난 폭 최대 (℃, %p, CP 단위) |
| `OP_ONLY`, `CP_ONLY` | 출력만 넘었나, CP만 넘었나 |
| `OTHER_ZONES` | 묶음 앞뒤 5분 안에 경보가 있던 다른 구역 수 |
| `TC_OP_SAME` | 묶음 앞뒤 5분 동안 온도와 출력이 같은 방향으로 움직인 비율 (정상 제어면 반대 방향) |
| `TC_Z_AFTER10` | 묶음 끝에서 10분 뒤 온도 \|Z\| (되돌아왔나) |
| `LEN`, `ELAPSED` | 묶음 길이(분), LOT 시작부터 묶음까지 지난 분 |
| `Z_구역`, `F_계열` | 구역 원핫, 강종 계열 원핫 |

In [ ]:
lot_start = df.groupby("LOT_NO")["MEAS_DT"].min()
lot_first_index = {}
lot_last_index = {}
for lot in lots:
    lot_index = df.index[df["LOT_NO"] == lot]
    lot_first_index[lot] = lot_index.min()
    lot_last_index[lot] = lot_index.max()

feature_rows = []
for i in range(len(candidates)):
    row = candidates.loc[i]
    zone = row["ZONE"]
    run = df.loc[row["FIRST"]:row["LAST"]]

    tc_sensors = []
    op_sensors = []
    cp_sensors = []
    for sensor in GROUPS[zone]:
        if sensor.startswith("OP-"):
            op_sensors.append(sensor)
        elif sensor.startswith("CP-"):
            cp_sensors.append(sensor)
        else:
            tc_sensors.append(sensor)

    feat = {"FAMILY": FAMILY.get(run["GRADE"].iloc[0], "OTHER"),
            "ELAPSED": (row["START"] - lot_start[row["LOT_NO"]]).total_seconds() / 60}

    # 센서 종류별 지난 값 |Z| 최대와 벗어난 폭 최대
    for name, sensor_list in [("TC", tc_sensors), ("OP", op_sensors), ("CP", cp_sensors)]:
        zmax = 0.0
        dev = 0.0
        for sensor in sensor_list:
            zmax = max(zmax, run[sensor + "__ZP"].max())
            dev = max(dev, run[sensor + "__DEVP"].max())
        feat[name + "_ZMAX"] = zmax
        feat[name + "_DEV"] = dev

    feat["OP_ONLY"] = int(feat["OP_ZMAX"] > Z_LIMIT and feat["TC_ZMAX"] <= Z_LIMIT and feat["CP_ZMAX"] <= Z_LIMIT)
    feat["CP_ONLY"] = int(feat["CP_ZMAX"] > Z_LIMIT and feat["TC_ZMAX"] <= Z_LIMIT and feat["OP_ZMAX"] <= Z_LIMIT)

    # 앞뒤 5분 안에 경보가 있던 다른 구역 수
    low = max(row["FIRST"] - NEAR_MINUTES, lot_first_index[row["LOT_NO"]])
    high = min(row["LAST"] + NEAR_MINUTES, lot_last_index[row["LOT_NO"]])
    other_zones = []
    for other in ZONES:
        if other != zone:
            other_zones.append(other)
    feat["OTHER_ZONES"] = int(alarm.loc[low:high, other_zones].any(axis=0).sum())

    # 온도와 출력이 같은 방향으로 움직인 비율
    if len(tc_sensors) > 0 and len(op_sensors) > 0:
        window = df.loc[low:high]
        same = ((window[tc_sensors[0]].diff() * window[op_sensors[0]].diff()) > 0).mean()
        feat["TC_OP_SAME"] = 0.0 if pd.isna(same) else float(same)
    else:
        feat["TC_OP_SAME"] = 0.0

    # 묶음 끝에서 10분 뒤 온도 |Z|
    after_index = min(row["LAST"] + AFTER_MINUTES, lot_last_index[row["LOT_NO"]])
    after_z = 0.0
    for sensor in tc_sensors:
        after_z = max(after_z, df.loc[after_index, sensor + "__ZP"])
    feat["TC_Z_AFTER10"] = after_z

    feature_rows.append(feat)

features = pd.DataFrame(feature_rows)
candidates = pd.concat([candidates, features], axis=1)

for zone in ZONES:
    candidates["Z_" + zone] = (candidates["ZONE"] == zone).astype(int)
for family in FAMILIES:
    candidates["F_" + family] = (candidates["FAMILY"] == family).astype(int)

FEATURES = ["LEN", "ELAPSED", "TC_ZMAX", "TC_DEV", "OP_ZMAX", "OP_DEV", "CP_ZMAX", "CP_DEV", "OP_ONLY", "CP_ONLY", "OTHER_ZONES", "TC_OP_SAME", "TC_Z_AFTER10"]
for zone in ZONES:
    FEATURES.append("Z_" + zone)
for family in FAMILIES:
    FEATURES.append("F_" + family)

labeled = candidates[candidates["LABEL"].notna()].copy()
labeled["LABEL"] = labeled["LABEL"].astype(int)
unlabeled = candidates[candidates["LABEL"].isna()].copy()
print("학습에 쓰는 후보:", len(labeled), "/ 팀이 안 본 후보:", len(unlabeled))

## 8. LOT 하나 빼고 학습: RandomForest가 팀 판단을 얼마나 배우나

In [ ]:
# RandomForest 설정 (1분 단위 실험에서 나무를 끝까지 키운 설정이 나았음)
N_TREES = 300
MIN_LEAF = 2

labeled["RF_PRED"] = 0
labeled["RF_PROB"] = 0.0

for lot in lots:
    train = labeled[labeled["LOT_NO"] != lot]
    test_rows = labeled["LOT_NO"] == lot

    if test_rows.sum() == 0 or train["LABEL"].sum() == 0:
        continue

    rf = RandomForestClassifier(n_estimators=N_TREES, max_depth=None, min_samples_leaf=MIN_LEAF, class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(train[FEATURES], train["LABEL"])

    labeled.loc[test_rows, "RF_PRED"] = rf.predict(labeled.loc[test_rows, FEATURES])
    labeled.loc[test_rows, "RF_PROB"] = rf.predict_proba(labeled.loc[test_rows, FEATURES])[:, 1]

# 비교할 방법: RF, 규칙 둘, 모든 후보를 1로 (지금 Z-score 그대로)
methods = {
    "RandomForest (후보 단위)": labeled["RF_PRED"],
    "규칙: CP 지난 값 Z > 3이면 1": (labeled["CP_ZMAX"] > Z_LIMIT).astype(int),
    "규칙: CP만 넘었으면 1": labeled["CP_ONLY"],
    "모든 후보를 1로 (지금 Z-score)": pd.Series(1, index=labeled.index),
}

rows = []
for name in methods:
    y_true = labeled["LABEL"]
    y_pred = methods[name]

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())

    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

    rows.append({"방법": name, "팀 1 맞힘": "%d/%d" % (tp, tp + fn), "1이라 한 것 중 진짜": round(precision, 2),
                 "팀 0 중 걸러냄": "%d/%d" % (tn, tn + fp), "걸러낸 비율": round(tn / (tn + fp), 2) if tn + fp > 0 else 0.0, "F1": round(f1, 3)})

leave_one_out = pd.DataFrame(rows)
leave_one_out

## 9. 시간순 학습 (앞 LOT으로만 학습, 03과 같은 방향)

In [ ]:
labeled["RF_PRED_TIME"] = np.nan

for k in range(1, len(lots)):
    train = labeled[labeled["LOT_NO"].isin(lots[:k])]
    test_rows = labeled["LOT_NO"] == lots[k]

    if test_rows.sum() == 0 or train["LABEL"].sum() == 0:
        continue

    rf = RandomForestClassifier(n_estimators=N_TREES, max_depth=None, min_samples_leaf=MIN_LEAF, class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(train[FEATURES], train["LABEL"])

    labeled.loc[test_rows, "RF_PRED_TIME"] = rf.predict(labeled.loc[test_rows, FEATURES])

scored = labeled[labeled["RF_PRED_TIME"].notna()]
y_true = scored["LABEL"]
y_pred = scored["RF_PRED_TIME"].astype(int)

tp = int(((y_true == 1) & (y_pred == 1)).sum())
fp = int(((y_true == 0) & (y_pred == 1)).sum())
fn = int(((y_true == 1) & (y_pred == 0)).sum())
tn = int(((y_true == 0) & (y_pred == 0)).sum())

print("평가한 후보:", len(scored), "(1:", int(y_true.sum()), ")")
print("팀 1 맞힘 %d/%d, 1이라 한 것 중 진짜 %.2f, 팀 0 중 걸러냄 %d/%d" % (tp, tp + fn, tp / (tp + fp) if tp + fp > 0 else 0, tn, tn + fp))

## 10. 팀 1 후보별 판단과 어긋난 후보

In [ ]:
show_columns = ["ZONE", "LOT_NO", "START", "LEN", "DECISIONS", "CP_ZMAX", "TC_ZMAX", "OP_ZMAX", "OTHER_ZONES", "RF_PRED", "RF_PROB"]

print("팀 1 후보 (LOT 하나 빼고 학습한 RF의 판단):")
display(labeled.loc[labeled["LABEL"] == 1, show_columns].round(2))

print("RF는 1이라 했는데 팀은 0인 후보 (팀이 다시 볼 만한 것):")
labeled.loc[(labeled["LABEL"] == 0) & (labeled["RF_PRED"] == 1), show_columns].round(2)

## 11. 조업 조건 특징: 강종변경 뒤 경과 분

- 조업이벤트(G-02)의 `강종변경` 시각과 LOT 시작 시각을 맞춰 봄. 맞으면 `ELAPSED`(LOT 시작부터 지난 분)가 곧 강종을 바꾸고 지난 분임
- 이벤트 시각 기준으로 `GRADE_ELAPSED`(마지막 강종변경부터 후보까지 지난 분)를 따로 만들고, 조업 조건 특징(경과 분, 강종 계열)을 뺐을 때와 넣었을 때를 LOT 하나 빼고 학습으로 비교함


In [ ]:
events = pd.read_csv(EVENT_CSV, encoding=ENC)
events["EVT_DT"] = pd.to_datetime(events["EVT_DT"], errors="raise")

grade_change = events[(events["EVT_TYPE"] == "강종변경") & (events["PLANT_CD"] == "CR1")].sort_values("EVT_DT").reset_index(drop=True)

# LOT 시작 시각과 가장 가까운 강종변경 이벤트
rows = []
for lot in lots:
    start = lot_start[lot]
    gap = (grade_change["EVT_DT"] - start).abs()
    nearest = gap.idxmin()

    rows.append({"LOT_NO": lot, "GRADE": df.loc[df["LOT_NO"] == lot, "GRADE"].iloc[0], "LOT 시작": start,
                 "가장 가까운 강종변경": grade_change.loc[nearest, "EVT_DT"], "바뀐 강종": grade_change.loc[nearest, "AFT_VAL"],
                 "차이(분)": round(gap[nearest].total_seconds() / 60, 1)})

lot_events = pd.DataFrame(rows)
print("LOT 시작이 강종변경 이벤트와 5분 안에 맞는 LOT:", int((lot_events["차이(분)"] <= 5).sum()), "/", len(lots))
lot_events


In [ ]:
# 후보 시작 시각 기준으로 마지막 강종변경부터 지난 분 (이벤트 시각 기준). 앞에 이벤트가 없으면 LOT 시작 기준(ELAPSED)을 씀
grade_elapsed = []
for i in range(len(candidates)):
    start = candidates.loc[i, "START"]
    before = grade_change[grade_change["EVT_DT"] <= start]

    if len(before) == 0:
        grade_elapsed.append(candidates.loc[i, "ELAPSED"])
    else:
        grade_elapsed.append((start - before["EVT_DT"].iloc[-1]).total_seconds() / 60)

candidates["GRADE_ELAPSED"] = grade_elapsed
labeled["GRADE_ELAPSED"] = candidates.loc[labeled.index, "GRADE_ELAPSED"]
unlabeled["GRADE_ELAPSED"] = candidates.loc[unlabeled.index, "GRADE_ELAPSED"]

different = (candidates["GRADE_ELAPSED"] - candidates["ELAPSED"]).abs() > 1
print("ELAPSED와 GRADE_ELAPSED가 1분 넘게 다른 후보:", int(different.sum()), "/ LOT:", [int(lot) for lot in sorted(candidates.loc[different, "LOT_NO"].unique())])


In [ ]:
# 조업 조건 특징을 뺐을 때와 넣었을 때 (LOT 하나 빼고 학습, 8절과 같은 설정)
family_columns = []
for family in FAMILIES:
    family_columns.append("F_" + family)

no_condition = []
for column in FEATURES:
    if column != "ELAPSED" and column not in family_columns:
        no_condition.append(column)

feature_sets = {
    "조업 조건 없음": no_condition,
    "+ LOT 시작부터 지난 분 (ELAPSED)": no_condition + ["ELAPSED"],
    "+ 강종변경부터 지난 분 (GRADE_ELAPSED)": no_condition + ["GRADE_ELAPSED"],
    "+ ELAPSED와 강종 계열 (지금 FEATURES)": FEATURES,
    "+ GRADE_ELAPSED와 강종 계열": no_condition + ["GRADE_ELAPSED"] + family_columns,
}

rows = []
for name in feature_sets:
    columns = feature_sets[name]
    pred = pd.Series(0, index=labeled.index)

    for lot in lots:
        train = labeled[labeled["LOT_NO"] != lot]
        test_rows = labeled["LOT_NO"] == lot

        if test_rows.sum() == 0 or train["LABEL"].sum() == 0:
            continue

        rf = RandomForestClassifier(n_estimators=N_TREES, max_depth=None, min_samples_leaf=MIN_LEAF, class_weight="balanced", random_state=42, n_jobs=-1)
        rf.fit(train[columns], train["LABEL"])
        pred.loc[test_rows] = rf.predict(labeled.loc[test_rows, columns])

    y_true = labeled["LABEL"]
    tp = int(((y_true == 1) & (pred == 1)).sum())
    fp = int(((y_true == 0) & (pred == 1)).sum())
    fn = int(((y_true == 1) & (pred == 0)).sum())
    tn = int(((y_true == 0) & (pred == 0)).sum())

    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

    rows.append({"특징 묶음": name, "특징 수": len(columns), "팀 1 맞힘": "%d/%d" % (tp, tp + fn), "1이라 한 것 중 진짜": round(precision, 2),
                 "팀 0 중 걸러냄": "%d/%d" % (tn, tn + fp), "F1": round(f1, 3)})

condition_compare = pd.DataFrame(rows)
condition_compare


In [ ]:
# 지금 FEATURES로 전체를 학습했을 때 조업 조건 특징의 중요도 (전체 합 1)
rf = RandomForestClassifier(n_estimators=N_TREES, max_depth=None, min_samples_leaf=MIN_LEAF, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(labeled[FEATURES], labeled["LABEL"])

importance = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("전체 상위 6:", ", ".join(importance.head(6).index))
print("강종 계열 원핫 합:", round(float(importance[family_columns].sum()), 3))
importance[["ELAPSED"] + family_columns].round(3)


## 12. 학습 범위 비교: 초반 LOT, 후반 LOT, 같은 강종 계열

- 초반 7개 LOT으로 학습해 후반 7개 LOT을 평가 (시간을 거슬러 쓰지 않는 가장 엄격한 시험). 반대 방향은 참고
- 같은 강종 계열 LOT으로만 학습 (LOT 하나 빼고) vs 전체 LOT으로 학습 (8절 `RF_PRED`). 학습에 쓸 같은 계열 LOT에 팀 1이 없으면 그 LOT은 평가하지 못함


In [ ]:
early_lots = lots[:7]
late_lots = lots[7:]

rows = []
for name, train_lots, test_lots in [("초반 7개 → 후반 7개", early_lots, late_lots), ("후반 7개 → 초반 7개 (참고)", late_lots, early_lots)]:
    train = labeled[labeled["LOT_NO"].isin(train_lots)]
    test = labeled[labeled["LOT_NO"].isin(test_lots)]

    rf = RandomForestClassifier(n_estimators=N_TREES, max_depth=None, min_samples_leaf=MIN_LEAF, class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(train[FEATURES], train["LABEL"])
    y_pred = pd.Series(rf.predict(test[FEATURES]), index=test.index)
    y_true = test["LABEL"]

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())

    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

    rows.append({"학습 → 평가": name, "학습 후보 (팀 1)": "%d (%d)" % (len(train), int(train["LABEL"].sum())), "평가 후보 (팀 1)": "%d (%d)" % (len(test), int(test["LABEL"].sum())),
                 "팀 1 맞힘": "%d/%d" % (tp, tp + fn), "1이라 한 것 중 진짜": round(precision, 2), "팀 0 중 걸러냄": "%d/%d" % (tn, tn + fp), "F1": round(f1, 3)})

scope_compare = pd.DataFrame(rows)
print("초반 LOT:", [int(lot) for lot in early_lots])
print("후반 LOT:", [int(lot) for lot in late_lots])
scope_compare


In [ ]:
# 같은 강종 계열 LOT으로만 학습 (LOT 하나 빼고)
lot_family = candidates.groupby("LOT_NO")["FAMILY"].first()

labeled["RF_PRED_FAMILY"] = np.nan
for lot in lots:
    train = labeled[(labeled["LOT_NO"] != lot) & (labeled["FAMILY"] == lot_family[lot])]
    test_rows = labeled["LOT_NO"] == lot

    if test_rows.sum() == 0 or len(train) == 0 or train["LABEL"].sum() == 0:
        continue

    rf = RandomForestClassifier(n_estimators=N_TREES, max_depth=None, min_samples_leaf=MIN_LEAF, class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(train[FEATURES], train["LABEL"])
    labeled.loc[test_rows, "RF_PRED_FAMILY"] = rf.predict(labeled.loc[test_rows, FEATURES])

# 계열별로 같은 계열만 학습한 결과와 전체 LOT으로 학습한 결과(8절)를 나란히. 전체 LOT 쪽은 같은 계열 쪽이 평가한 후보만 셈
rows = []
for family in FAMILIES:
    group = labeled[labeled["FAMILY"] == family]
    scored = group[group["RF_PRED_FAMILY"].notna()]

    tp_same = int(((scored["LABEL"] == 1) & (scored["RF_PRED_FAMILY"] == 1)).sum())
    fn_same = int(((scored["LABEL"] == 1) & (scored["RF_PRED_FAMILY"] == 0)).sum())
    tn_same = int(((scored["LABEL"] == 0) & (scored["RF_PRED_FAMILY"] == 0)).sum())
    fp_same = int(((scored["LABEL"] == 0) & (scored["RF_PRED_FAMILY"] == 1)).sum())

    tp_all = int(((scored["LABEL"] == 1) & (scored["RF_PRED"] == 1)).sum())
    fn_all = int(((scored["LABEL"] == 1) & (scored["RF_PRED"] == 0)).sum())
    tn_all = int(((scored["LABEL"] == 0) & (scored["RF_PRED"] == 0)).sum())
    fp_all = int(((scored["LABEL"] == 0) & (scored["RF_PRED"] == 1)).sum())

    rows.append({"계열": family, "LOT 수": int((lot_family == family).sum()), "후보 (팀 1)": "%d (%d)" % (len(group), int(group["LABEL"].sum())),
                 "평가한 후보": len(scored), "같은 계열만: 팀 1 맞힘": "%d/%d" % (tp_same, tp_same + fn_same), "같은 계열만: 팀 0 걸러냄": "%d/%d" % (tn_same, tn_same + fp_same),
                 "전체 LOT: 팀 1 맞힘": "%d/%d" % (tp_all, tp_all + fn_all), "전체 LOT: 팀 0 걸러냄": "%d/%d" % (tn_all, tn_all + fp_all)})

family_compare = pd.DataFrame(rows)
print("LOT별 계열:", ", ".join(str(lot)[-3:] + " " + lot_family[lot] for lot in lots))
family_compare


## 13. 팀이 안 본 후보에 점수 매기기와 저장

In [ ]:
# 라벨 있는 후보 전체로 학습해 팀이 안 본 후보에 확률을 매김 (다음 검토 후보)
rf = RandomForestClassifier(n_estimators=N_TREES, max_depth=None, min_samples_leaf=MIN_LEAF, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(labeled[FEATURES], labeled["LABEL"])

unlabeled["RF_PROB"] = rf.predict_proba(unlabeled[FEATURES])[:, 1]
unlabeled = unlabeled.sort_values("RF_PROB", ascending=False)

importance = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("중요 특징:", ", ".join(importance.head(8).index))

save_columns = ["ZONE", "LOT_NO", "START", "END", "LEN", "LABEL", "DECISIONS", "FAMILY", "ELAPSED", "GRADE_ELAPSED",
                "TC_ZMAX", "TC_DEV", "OP_ZMAX", "OP_DEV", "CP_ZMAX", "CP_DEV", "OP_ONLY", "CP_ONLY", "OTHER_ZONES", "TC_OP_SAME", "TC_Z_AFTER10"]
candidates[save_columns].to_csv(OUT_CANDIDATES, index=False, encoding=ENC)
unlabeled[save_columns + ["RF_PROB"]].to_csv(OUT_RANKED, index=False, encoding=ENC)

print("저장:", OUT_CANDIDATES, "/", OUT_RANKED)
print("팀이 안 본 후보 중 RF 확률 0.5 이상:", int((unlabeled["RF_PROB"] >= 0.5).sum()), "/", len(unlabeled))
unlabeled[["ZONE", "LOT_NO", "START", "LEN", "CP_ZMAX", "TC_ZMAX", "TC_DEV", "OP_ZMAX", "OTHER_ZONES", "RF_PROB"]].head(12).round(2)